# Error Analysis and Extracted-Random Similarity

This notebook analyzes non-extracted selections and the relationship between extracted-random text similarity and annotator disagreement.


In [2]:
from pathlib import Path
from collections import Counter, defaultdict
import json
import math
import re

import pandas as pd
from openpyxl import load_workbook

base_dir = Path("/content")
answers_dir = base_dir
reference_path = base_dir / "narratives_clean.xlsx"
annotation_paths = [answers_dir / f"annotation_{i}.xlsx" for i in range(1, 6)]


def normalize(value):
    if value is None:
        return ""
    text = str(value)
    text = text.replace("’", "'").replace("‘", "'").replace("“", '"').replace("”", '"')
    return " ".join(text.split()).strip().lower()


def tokens(text):
    return re.findall(r"[a-zA-Z0-9']+", normalize(text))


def ngrams(words):
    out = list(words)
    out.extend(f"{words[i]} {words[i + 1]}" for i in range(len(words) - 1))
    return out


def classify(choice, ref):
    c = normalize(choice)
    if c == normalize(ref["narrative"]):
        return "extracted"
    if c == normalize(ref["random_narrative"]):
        return "random"
    if c == normalize(ref["false_narrative"]):
        return "false"
    if c == normalize("None of the above"):
        return "none"
    return "other"


def cosine(vec_a, vec_b):
    dot = sum(vec_a.get(k, 0.0) * vec_b.get(k, 0.0) for k in set(vec_a) | set(vec_b))
    norm_a = math.sqrt(sum(v * v for v in vec_a.values()))
    norm_b = math.sqrt(sum(v * v for v in vec_b.values()))
    if norm_a == 0 or norm_b == 0:
        return 0.0
    return dot / (norm_a * norm_b)


def pearson(xs, ys):
    mean_x = sum(xs) / len(xs)
    mean_y = sum(ys) / len(ys)
    num = sum((x - mean_x) * (y - mean_y) for x, y in zip(xs, ys))
    den_x = math.sqrt(sum((x - mean_x) ** 2 for x in xs))
    den_y = math.sqrt(sum((y - mean_y) ** 2 for y in ys))
    if den_x == 0 or den_y == 0:
        return 0.0
    return num / (den_x * den_y)


def ranks(values):
    indexed = sorted(enumerate(values), key=lambda x: x[1])
    out = [0.0] * len(values)
    i = 0
    while i < len(indexed):
        j = i
        while j + 1 < len(indexed) and indexed[j + 1][1] == indexed[i][1]:
            j += 1
        rank = (i + j + 2) / 2
        for k in range(i, j + 1):
            out[indexed[k][0]] = rank
        i = j + 1
    return out


def mean(values):
    return sum(values) / len(values) if values else 0.0


reference_df = pd.read_excel(reference_path)
refs = reference_df.loc[:99, ["narrative", "random_narrative", "false_narrative"]].to_dict("records")

texts = []
for ref in refs:
    texts.extend([ref["narrative"], ref["random_narrative"], ref["false_narrative"]])

doc_terms = [ngrams(tokens(text)) for text in texts]
df = Counter()
for terms in doc_terms:
    df.update(set(terms))

n_docs = len(doc_terms)
idf = {term: math.log((1 + n_docs) / (1 + count)) + 1 for term, count in df.items()}


def tfidf(text):
    counts = Counter(ngrams(tokens(text)))
    total = sum(counts.values())
    if total == 0:
        return {}
    return {term: (count / total) * idf[term] for term, count in counts.items()}


similarities = [
    cosine(tfidf(ref["narrative"]), tfidf(ref["random_narrative"]))
    for ref in refs
]

item_counts = []
annotator_rows = []

for item_index, ref in enumerate(refs):
    counts = Counter()
    by_annotator = {}
    for annotator_index, path in enumerate(annotation_paths, 1):
        ws = load_workbook(path, data_only=False).active
        choice = ws.cell(row=item_index + 2, column=4).value
        label = classify(choice, ref)
        counts[label] += 1
        by_annotator[f"A{annotator_index}"] = label
        annotator_rows.append({
            "row": item_index + 2,
            "annotator": f"A{annotator_index}",
            "label": label,
            "is_error": label != "extracted",
            "similarity_extracted_random": similarities[item_index],
        })
    item_counts.append({
        "row": item_index + 2,
        "similarity_extracted_random": similarities[item_index],
        "errors": 5 - counts["extracted"],
        "random_errors": counts["random"],
        "false_errors": counts["false"],
        "none_errors": counts["none"],
        "other_errors": counts["other"],
        "counts": dict(counts),
        "annotators": by_annotator,
    })

total_errors = sum(row["errors"] for row in item_counts)
total_random_errors = sum(row["random_errors"] for row in item_counts)
total_false_errors = sum(row["false_errors"] for row in item_counts)
total_none_errors = sum(row["none_errors"] for row in item_counts)
total_other_errors = sum(row["other_errors"] for row in item_counts)

error_items = [row for row in item_counts if row["errors"] > 0]
no_error_items = [row for row in item_counts if row["errors"] == 0]

xs = [row["similarity_extracted_random"] for row in item_counts]
ys = [row["errors"] for row in item_counts]
binary_ys = [1 if row["errors"] > 0 else 0 for row in item_counts]

by_error_count = defaultdict(list)
for row in item_counts:
    by_error_count[row["errors"]].append(row["similarity_extracted_random"])

top_error_rows = sorted(
    [row for row in item_counts if row["errors"] > 0],
    key=lambda row: (-row["errors"], -row["similarity_extracted_random"], row["row"]),
)

result = {
    "total_annotation_decisions": 500,
    "total_errors": total_errors,
    "error_breakdown": {
        "random": total_random_errors,
        "false": total_false_errors,
        "none": total_none_errors,
        "other": total_other_errors,
    },
    "items_with_any_error": len(error_items),
    "items_with_no_error": len(no_error_items),
    "mean_similarity_extracted_random_all_items": mean(xs),
    "mean_similarity_error_items": mean([row["similarity_extracted_random"] for row in error_items]),
    "mean_similarity_no_error_items": mean([row["similarity_extracted_random"] for row in no_error_items]),
    "mean_similarity_by_error_count": {
        str(k): mean(v) for k, v in sorted(by_error_count.items())
    },
    "pearson_similarity_vs_error_count": pearson(xs, ys),
    "spearman_similarity_vs_error_count": pearson(ranks(xs), ranks(ys)),
    "pearson_similarity_vs_any_error": pearson(xs, binary_ys),
    "spearman_similarity_vs_any_error": pearson(ranks(xs), ranks(binary_ys)),
    "top_error_rows": top_error_rows[:20],
}

print(json.dumps(result, indent=2))


{
  "total_annotation_decisions": 500,
  "total_errors": 77,
  "error_breakdown": {
    "random": 22,
    "false": 24,
    "none": 31,
    "other": 0
  },
  "items_with_any_error": 26,
  "items_with_no_error": 74,
  "mean_similarity_extracted_random_all_items": 0.016431809068844318,
  "mean_similarity_error_items": 0.02928875070295387,
  "mean_similarity_no_error_items": 0.011914505251454474,
  "mean_similarity_by_error_count": {
    "0": 0.011914505251454474,
    "1": 0.01879408216123906,
    "2": 0.04140186647992459,
    "3": 0.06227052814549039,
    "4": 0.015989143291293625,
    "5": 0.01899551958538006
  },
  "pearson_similarity_vs_error_count": 0.2215732731810007,
  "spearman_similarity_vs_error_count": 0.27746303396311955,
  "pearson_similarity_vs_any_error": 0.30387714475475974,
  "spearman_similarity_vs_any_error": 0.305844397452688,
  "top_error_rows": [
    {
      "row": 78,
      "similarity_extracted_random": 0.08065944535556806,
      "errors": 5,
      "random_errors": 